A edição impressa e o e-book de *Think Python 3e*, de Allen B. Downey, podem ser adquiridos na
[Bookshop.org](https://bookshop.org/a/98697/9781098155438) e na
[Amazon](https://www.amazon.com/_/dp/1098155432?smid=ATVPDKIKX0DER&_encoding=UTF8&tag=oreilly20-20&_encoding=UTF8&tag=greenteapre01-20&linkCode=ur2&linkId=e2a529f94920295d27ec8a06e757dc7c&camp=1789&creative=9325).

Esta tradução educacional em português brasileiro é gratuita, sem fins lucrativos, e destina-se a quem quer aprender lógica de programação em Python.

In [1]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');
download('https://github.com/ramalho/jupyturtle/releases/download/2024-03/jupyturtle.py');

import thinkpython

# Classes e objetos

Neste ponto já definimos classes e criamos objetos que representam a hora do dia e o dia do ano.
E definimos métodos que criam, modificam e fazem cálculos com esses objetos.

Neste capítulo vamos continuar o passeio pela programação orientada a objetos (POO) definindo classes que representam objetos geométricos, inclusive pontos, retas, retângulos e círculos.
Vamos escrever métodos que criam e modificam esses objetos, e usar o módulo `jupyturtle` para desenhá-los.

Vou usar essas classes para demonstrar tópicos de POO, inclusive identidade e equivalência de objetos, cópia rasa e cópia profunda, e polimorfismo.

## Criando um Point

Em computação gráfica, uma posição na tela costuma ser representada por um par de coordenadas em um plano `x`-`y`.
Por convenção, o ponto `(0, 0)` em geral representa o canto superior esquerdo da tela, e `(x, y)` representa o ponto `x` unidades à direita e `y` unidades abaixo da origem.
Em comparação com o sistema de coordenadas cartesianas que você pode ter visto numa aula de matemática, o eixo `y` está de cabeça para baixo.

Há várias formas de representar um ponto em Python:

-   Podemos guardar as coordenadas em duas variáveis separadas, `x` e `y`.

-   Podemos guardar as coordenadas como elementos de uma lista ou tupla.

-   Podemos criar um tipo novo para representar pontos como objetos.

Na programação orientada a objetos, o mais idiomático é criar um tipo novo.
Para isso, vamos começar com uma definição de classe para `Point`.

In [2]:
class Point:
    """Represents a point in 2-D space."""
    
    def __init__(self, x, y):
        self.x = x
        self.y = y
        
    def __str__(self):
        return f'Point({self.x}, {self.y})'

O método `__init__` recebe as coordenadas como parâmetros e as atribui aos atributos `x` e `y`.
O método `__str__` devolve uma representação em string do `Point`.

Agora podemos instanciar e exibir um objeto `Point` assim.

In [3]:
start = Point(0, 0)
print(start)

O diagrama a seguir mostra o estado do objeto novo.

In [4]:
from diagram import make_frame, make_binding

d1 = vars(start)
frame = make_frame(d1, name='Point', dy=-0.25, offsetx=0.18)
binding = make_binding('start', frame)

In [5]:
from diagram import diagram, adjust

width, height, x, y = [1.41, 0.89, 0.26, 0.5]
ax = diagram(width, height)
bbox = binding.draw(ax, x, y)
#adjust(x, y, bbox)

Como de costume, um tipo definido pelo programador é representado por uma caixa com o nome do tipo do lado de fora e os atributos do lado de dentro.

Em geral, tipos definidos pelo programador são mutáveis, então podemos escrever um método como `translate` que recebe dois números, `dx` e `dy`, e os soma aos atributos `x` e `y`.

In [6]:
%%add_method_to Point

    def translate(self, dx, dy):
        self.x += dx
        self.y += dy

Essa função translada o `Point` de um lugar do plano para outro.
Se não quisermos modificar um `Point` existente, podemos usar `copy` para copiar o objeto original e depois modificar a cópia.

In [7]:
from copy import copy

end1 = copy(start)
end1.translate(300, 0)
print(end1)

Podemos encapsular esses passos em outro método, chamado `translated`.

In [8]:
%%add_method_to Point

    def translated(self, dx=0, dy=0):
        point = copy(self)
        point.translate(dx, dy)
        return point

Do mesmo modo que o método `sort` modifica uma lista e a função `sorted` cria uma lista nova, agora temos um método `translate` que modifica um `Point` e um método `translated` que cria um novo.

Aqui vai um exemplo:

In [9]:
end2 = start.translated(0, 150)
print(end2)

Na próxima seção, vamos usar esses pontos para definir e desenhar uma reta.

## Criando uma Line

Agora vamos definir uma classe que representa o segmento de reta entre dois pontos.
Como de costume, vamos começar com um método `__init__` e um método `__str__`.

In [10]:
class Line:
    def __init__(self, p1, p2):
        self.p1 = p1
        self.p2 = p2
        
    def __str__(self):
        return f'Line({self.p1}, {self.p2})'

Com esses dois métodos, podemos instanciar e exibir um objeto `Line` que usaremos para representar o eixo `x`.

In [11]:
line1 = Line(start, end1)
print(line1)

Quando chamamos `print` e passamos `line` como parâmetro, `print` invoca `__str__` em `line`.
O método `__str__` usa uma f-string para criar uma representação em string da `line`.

A f-string contém duas expressões entre chaves, `self.p1` e `self.p2`.
Quando essas expressões são avaliadas, os resultados são objetos `Point`.
Depois, ao serem convertidos em strings, o método `__str__` da classe `Point` é invocado.

Por isso, quando exibimos uma `Line`, o resultado contém as representações em string dos objetos `Point`.

O diagrama de objeto a seguir mostra o estado deste objeto `Line`.

In [12]:
from diagram import Binding, Value, Frame

d1 = vars(line1.p1)
frame1 = make_frame(d1, name='Point', dy=-0.25, offsetx=0.17)

d2 = vars(line1.p2)
frame2 = make_frame(d2, name='Point', dy=-0.25, offsetx=0.17)

binding1 = Binding(Value('start'), frame1, dx=0.4)
binding2 = Binding(Value('end'), frame2, dx=0.4)
frame3 = Frame([binding1, binding2], name='Line', dy=-0.9, offsetx=0.4, offsety=-0.25)

binding = make_binding('line1', frame3)

In [13]:
width, height, x, y = [2.45, 2.12, 0.27, 1.76]
ax = diagram(width, height)
bbox = binding.draw(ax, x, y)
#adjust(x, y, bbox)

Representações em string e diagramas de objeto são úteis para depuração, mas o objetivo deste exemplo é gerar gráficos, não texto!
Então vamos usar o módulo `jupyturtle` para desenhar retas na tela.

Como fizemos no [capítulo 4](section_turtle_module), vamos usar `make_turtle` para criar um objeto `Turtle` e um canvas pequeno onde ele possa desenhar.
Para desenhar retas, vamos usar duas funções novas do módulo `jupyturtle`:

* `jumpto`, que recebe duas coordenadas e move a `Turtle` até o local indicado sem desenhar uma reta, e

* `moveto`, que move a `Turtle` da posição atual até o local indicado e desenha um segmento de reta entre eles.

Aqui está como as importamos.

In [14]:
from jupyturtle import make_turtle, jumpto, moveto

E aqui está um método que desenha uma `Line`.

In [15]:
%%add_method_to Line

    def draw(self):
        jumpto(self.p1.x, self.p1.y)
        moveto(self.p2.x, self.p2.y)

Para mostrar como ele é usado, vou criar uma segunda reta que representa o eixo `y`.

In [16]:
line2 = Line(start, end2)
print(line2)

E depois desenhar os eixos.

In [17]:
make_turtle()
line1.draw()
line2.draw()

À medida que definirmos e desenharmos mais objetos, vamos usar essas retas de novo.
Mas primeiro vamos falar de equivalência e identidade de objetos.

## Equivalência e identidade

Suponha que criemos dois pontos com as mesmas coordenadas.

In [18]:
p1 = Point(200, 100)
p2 = Point(200, 100)

Se usarmos o operador `==` para compará-los, obtemos o comportamento padrão dos tipos definidos pelo programador: o resultado é `True` só se forem o mesmo objeto, o que não é o caso.

In [19]:
p1 == p2

Se quisermos mudar esse comportamento, podemos fornecer um método especial chamado `__eq__` que define o que significa dois objetos `Point` serem iguais.

In [20]:
%%add_method_to Point

def __eq__(self, other):
    return (self.x == other.x) and (self.y == other.y)

Essa definição considera dois `Points` iguais se os atributos forem iguais.
Agora, quando usamos o operador `==`, ele invoca o método `__eq__`, que indica que `p1` e `p2` são considerados iguais.

In [21]:
p1 == p2

Mas o operador `is` ainda indica que são objetos diferentes.

In [22]:
p1 is p2

Não é possível sobrescrever o operador `is`: ele sempre verifica se os objetos são idênticos.
Mas, para tipos definidos pelo programador, você pode sobrescrever o operador `==` para que verifique se os objetos são equivalentes.
E você pode definir o que equivalente significa.

## Criando um Rectangle

Agora vamos definir uma classe que representa e desenha retângulos.
Para manter as coisas simples, vamos supor que os retângulos sejam verticais ou horizontais, não inclinados.
Quais atributos você acha que deveríamos usar para especificar a posição e o tamanho de um retângulo?

Há pelo menos duas possibilidades:

-   Você poderia especificar a largura e a altura do retângulo e a posição de um canto.

-   Você poderia especificar dois cantos opostos.

Neste ponto é difícil dizer se uma é melhor que a outra, então vamos implementar a primeira.
Aqui está a definição da classe.

In [23]:
class Rectangle:
    """Represents a rectangle. 

    attributes: width, height, corner.
    """
    def __init__(self, width, height, corner):
        self.width = width
        self.height = height
        self.corner = corner
        
    def __str__(self):
        return f'Rectangle({self.width}, {self.height}, {self.corner})'

Como de costume, o método `__init__` atribui os parâmetros aos atributos e o `__str__` devolve uma representação em string do objeto.
Agora podemos instanciar um objeto `Rectangle`, usando um `Point` como posição do canto superior esquerdo.

In [24]:
corner = Point(30, 20)
box1 = Rectangle(100, 50, corner)
print(box1)

O diagrama a seguir mostra o estado deste objeto.

In [25]:
from diagram import Binding, Value

def make_rectangle_binding(name, box, **options):
    d1 = vars(box.corner)
    frame_corner = make_frame(d1, name='Point', dy=-0.25, offsetx=0.07)

    d2 = dict(width=box.width, height=box.height)
    frame = make_frame(d2, name='Rectangle', dy=-0.25, offsetx=0.45)
    binding = Binding(Value('corner'), frame1, dx=0.92, draw_value=False, **options)
    frame.bindings.append(binding)

    binding = Binding(Value(name), frame)
    return binding, frame_corner

binding_box1, frame_corner1 = make_rectangle_binding('box1', box1)

In [26]:
from diagram import Bbox

width, height, x, y = [2.83, 1.49, 0.27, 1.1]
ax = diagram(width, height)
bbox1 = binding_box1.draw(ax, x, y)
bbox2 = frame_corner1.draw(ax, x+1.85, y-0.6)
bbox = Bbox.union([bbox1, bbox2])
#adjust(x, y, bbox)

Para desenhar um retângulo, vamos usar o método a seguir para criar quatro objetos `Point` que representam os cantos.

In [27]:
%%add_method_to Rectangle

    def make_points(self):
        p1 = self.corner
        p2 = p1.translated(self.width, 0)
        p3 = p2.translated(0, self.height)
        p4 = p3.translated(-self.width, 0)
        return p1, p2, p3, p4

Depois vamos criar quatro objetos `Line` que representam os lados.

In [28]:
%%add_method_to Rectangle

    def make_lines(self):
        p1, p2, p3, p4 = self.make_points()
        return Line(p1, p2), Line(p2, p3), Line(p3, p4), Line(p4, p1)

Depois vamos desenhar os lados.

In [29]:
%%add_method_to Rectangle

    def draw(self):
        lines = self.make_lines()
        for line in lines:
            line.draw()

Aqui vai um exemplo.

In [30]:
make_turtle()
line1.draw()
line2.draw()
box1.draw()

A figura inclui duas retas para representar os eixos.

## Alterando retângulos

Agora vamos considerar dois métodos que modificam retângulos, `grow` e `translate`.
Vamos ver que `grow` funciona como esperado, mas `translate` tem um bug sutil.
Tente descobrir qual é antes de eu explicar.

`grow` recebe dois números, `dwidth` e `dheight`, e os soma aos atributos `width` e `height` do retângulo.

In [31]:
%%add_method_to Rectangle

    def grow(self, dwidth, dheight):
        self.width += dwidth
        self.height += dheight

Aqui vai um exemplo que demonstra o efeito fazendo uma cópia de `box1` e invocando `grow` na cópia.

In [32]:
box2 = copy(box1)
box2.grow(60, 40)
print(box2)

Se desenharmos `box1` e `box2`, podemos confirmar que `grow` funciona como esperado.

In [33]:
make_turtle()
line1.draw()
line2.draw()
box1.draw()
box2.draw()

Agora vamos ver o `translate`.
Ele recebe dois números, `dx` e `dy`, e move o retângulo essas distâncias nas direções `x` e `y`.

In [34]:
%%add_method_to Rectangle

    def translate(self, dx, dy):
        self.corner.translate(dx, dy)

Para demonstrar o efeito, vamos transladar `box2` para a direita e para baixo.

In [35]:
box2.translate(30, 20)
print(box2)

Agora vamos ver o que acontece se desenharmos `box1` e `box2` de novo.

In [36]:
make_turtle()
line1.draw()
line2.draw()
box1.draw()
box2.draw()

Parece que os dois retângulos se moveram, o que não era a intenção!
A próxima seção explica o que deu errado.

## Cópia profunda

Quando usamos `copy` para duplicar `box1`, ele copia o objeto `Rectangle`, mas não o objeto `Point` que ele contém.
Então `box1` e `box2` são objetos diferentes, como pretendido.

In [37]:
box1 is box2

Mas os atributos `corner` deles se referem ao mesmo objeto.

In [38]:
box1.corner is box2.corner

O diagrama a seguir mostra o estado desses objetos.

In [39]:
from diagram import Stack
from copy import deepcopy

binding_box1, frame_corner1 = make_rectangle_binding('box1', box1)
binding_box2, frame_corner2 = make_rectangle_binding('box2', box2, dy=0.4)
binding_box2.value.bindings.reverse()

stack = Stack([binding_box1, binding_box2], dy=-1.3)

In [40]:
from diagram import Bbox

width, height, x, y = [2.76, 2.54, 0.27, 2.16]
ax = diagram(width, height)
bbox1 = stack.draw(ax, x, y)
bbox2 = frame_corner1.draw(ax, x+1.85, y-0.6)
bbox = Bbox.union([bbox1, bbox2])
# adjust(x, y, bbox)

O que `copy` faz se chama **cópia rasa**, porque copia o objeto, mas não os objetos que ele contém.
Como resultado, mudar o `width` ou o `height` de um `Rectangle` não afeta o outro, mas mudar os atributos do `Point` compartilhado afeta os dois!
Esse comportamento é confuso e sujeito a erros.

Por sorte, o módulo `copy` oferece outra função, chamada `deepcopy`, que copia não só o objeto, mas também os objetos a que ele se refere, e os objetos a que *eles* se referem, e assim por diante.
Essa operação se chama **cópia profunda**.

Para demonstrar, vamos começar com um `Rectangle` novo que contém um `Point` novo.

In [41]:
corner = Point(20, 20)
box3 = Rectangle(100, 50, corner)
print(box3)

E vamos fazer uma cópia profunda.

In [42]:
from copy import deepcopy

box4 = deepcopy(box3)

Podemos confirmar que os dois objetos `Rectangle` se referem a objetos `Point` diferentes.

In [43]:
box3.corner is box4.corner

Como `box3` e `box4` são objetos completamente separados, podemos modificar um sem afetar o outro.
Para demonstrar, vamos mover `box3` e aumentar `box4`.

In [44]:
box3.translate(50, 30)
box4.grow(100, 60)

E podemos confirmar que o efeito é o esperado.

In [45]:
make_turtle()
line1.draw()
line2.draw()
box3.draw()
box4.draw()

## Polimorfismo

No exemplo anterior, invocamos o método `draw` em dois objetos `Line` e dois objetos `Rectangle`.
Podemos fazer a mesma coisa de forma mais concisa criando uma lista de objetos.

In [46]:
shapes = [line1, line2, box3, box4]

Os elementos dessa lista são de tipos diferentes, mas todos oferecem um método `draw`, então podemos percorrer a lista e invocar `draw` em cada um.

In [47]:
make_turtle()

for shape in shapes:
    shape.draw()

Na primeira e na segunda passagem pelo laço, `shape` se refere a um objeto `Line`, então, quando `draw` é invocado, o método que roda é o definido na classe `Line`.

Na terceira e na quarta passagem pelo laço, `shape` se refere a um objeto `Rectangle`, então, quando `draw` é invocado, o método que roda é o definido na classe `Rectangle`.

Num certo sentido, cada objeto sabe como se desenhar.
Esse recurso se chama **polimorfismo**.
A palavra vem de raízes gregas que significam "muitas formas".
Na programação orientada a objetos, polimorfismo é a capacidade de tipos diferentes oferecerem os mesmos métodos, o que torna possível realizar muitos cálculos, como desenhar formas, invocando o mesmo método em tipos diferentes de objetos.

Como exercício no fim deste capítulo, você vai definir uma classe nova que representa um círculo e oferece um método `draw`.
Depois poderá usar polimorfismo para desenhar retas, retângulos e círculos.

## Depuração

Neste capítulo, esbarramos num bug sutil que aconteceu porque criamos um `Point` compartilhado por dois objetos `Rectangle` e depois modificamos o `Point`.
Em geral, há duas formas de evitar problemas assim: você pode evitar compartilhar objetos ou evitar modificá-los.

Para evitar compartilhar objetos, você pode usar cópia profunda, como fizemos neste capítulo.

Para evitar modificar objetos, considere substituir funções impuras como `translate` por funções puras como `translated`.
Por exemplo, aqui está uma versão de `translated` que cria um `Point` novo e nunca modifica seus atributos.

In [48]:
    def translated(self, dx=0, dy=0):
        x = self.x + dx
        y = self.y + dy
        return Point(x, y)

Python oferece recursos que tornam mais fácil evitar modificar objetos.
Eles estão além do escopo deste livro, mas, se tiver curiosidade, pergunte a um assistente virtual: "How do I make a Python object immutable?"

Criar um objeto novo leva mais tempo do que modificar um existente, mas a diferença raramente importa na prática.
Programas que evitam objetos compartilhados e funções impuras costumam ser mais fáceis de desenvolver, testar e depurar, e o melhor tipo de depuração é aquele que você não precisa fazer.

## Glossário

**cópia rasa:**
Operação de cópia que não copia objetos aninhados.

**cópia profunda:**
Operação de cópia que também copia objetos aninhados.

**polimorfismo:**
Capacidade de um método ou operador funcionar com vários tipos de objetos.

## Exercícios

In [ ]:
# Esta célula pede ao Jupyter informações detalhadas de depuração
# quando ocorre um erro em tempo de execução. Execute-a antes dos exercícios.

%xmode Verbose

### Pergunte a um assistente virtual

Em todos os exercícios a seguir, considere pedir ajuda a um assistente virtual.
Se fizer isso, inclua no prompt as definições das classes `Point`, `Line` e `Rectangle`. Caso contrário, o assistente vai chutar os atributos e as funções, e o código gerado não vai funcionar.

### Exercício

Escreva um método `__eq__` para a classe `Line` que devolva `True` se os objetos `Line` se referirem a objetos `Point` equivalentes, em qualquer ordem.

Você pode usar o esboço a seguir para começar.

In [49]:
%%add_method_to Line

def __eq__(self, other):
    return None

In [50]:
# A solução vai aqui

Você pode usar estes exemplos para testar o código.

In [51]:
start1 = Point(0, 0)
start2 = Point(0, 0)
end = Point(200, 100)

Este exemplo deve ser `True` porque os objetos `Line` se referem a objetos `Point` equivalentes, na mesma ordem.

In [52]:
line_a = Line(start1, end)
line_b = Line(start2, end)
line_a == line_b    # deve ser True

Este exemplo deve ser `True` porque os objetos `Line` se referem a objetos `Point` equivalentes, na ordem inversa.

In [53]:
line_c = Line(end, start1)
line_a == line_c     # deve ser True

A equivalência deve ser sempre transitiva: isto é, se `line_a` e `line_b` forem equivalentes, e `line_a` e `line_c` forem equivalentes, então `line_b` e `line_c` também devem ser equivalentes.

In [54]:
line_b == line_c     # deve ser True

Este exemplo deve ser `False` porque os objetos `Line` se referem a objetos `Point` que não são equivalentes.

In [55]:
line_d = Line(start1, start2)
line_a == line_d    # deve ser False

### Exercício

Escreva um método de `Line` chamado `midpoint` que calcule o ponto médio de um segmento de reta e devolva o resultado como um objeto `Point`.

Você pode usar o esboço a seguir para começar.

In [56]:
%%add_method_to Line

    def midpoint(self):
        return Point(0, 0)

In [57]:
# A solução vai aqui

Você pode usar os exemplos a seguir para testar o código e desenhar o resultado.

In [58]:
start = Point(0, 0)
end1 = Point(300, 0)
end2 = Point(0, 150)
line1 = Line(start, end1)
line2 = Line(start, end2)

In [59]:
mid1 = line1.midpoint()
print(mid1)

In [60]:
mid2 = line2.midpoint()
print(mid2)

In [61]:
line3 = Line(mid1, mid2)

In [62]:
make_turtle()

for shape in [line1, line2, line3]:
    shape.draw()

### Exercício

Escreva um método de `Rectangle` chamado `midpoint` que encontre o ponto no centro de um retângulo e devolva o resultado como um objeto `Point`.

Você pode usar o esboço a seguir para começar.

In [63]:
%%add_method_to Rectangle

    def midpoint(self):
        return Point(0, 0)

In [64]:
# A solução vai aqui

Você pode usar o exemplo a seguir para testar o código.

In [65]:
corner = Point(30, 20)
rectangle = Rectangle(100, 80, corner)

In [66]:
mid = rectangle.midpoint()
print(mid)

In [67]:
diagonal = Line(corner, mid)

In [68]:
make_turtle()

for shape in [line1, line2, rectangle, diagonal]:
    shape.draw()

### Exercício

Escreva um método de `Rectangle` chamado `make_cross` que:

1. Use `make_lines` para obter uma lista de objetos `Line` que representem os quatro lados do retângulo.

2. Calcule os pontos médios das quatro retas.

3. Crie e devolva uma lista de dois objetos `Line` que representem retas ligando pontos médios opostos, formando uma cruz pelo meio do retângulo.

Você pode usar este esboço para começar.

In [69]:
%%add_method_to Rectangle

    def make_diagonals(self):
        return []

In [70]:
# A solução vai aqui

Você pode usar o exemplo a seguir para testar o código.

In [71]:
corner = Point(30, 20)
rectangle = Rectangle(100, 80, corner)

In [72]:
lines = rectangle.make_cross()

In [73]:
make_turtle()

rectangle.draw()
for line in lines:
    line.draw()

### Exercício

Escreva uma definição para uma classe chamada `Circle` com atributos `center` e `radius`, em que `center` é um objeto `Point` e `radius` é um número.
Inclua os métodos especiais `__init__` e `__str__`, e um método chamado `draw` que use funções de `jupyturtle` para desenhar o círculo.

Você pode usar a função a seguir, que é uma versão da função `circle` que escrevemos no capítulo 4.

In [74]:
from jupyturtle import make_turtle, forward, left, right
import math
    
def draw_circle(radius):
    circumference = 2 * math.pi * radius
    n = 30
    length = circumference / n
    angle = 360 / n
    left(angle / 2)
    for i in range(n):
        forward(length)
        left(angle)

In [75]:
# A solução vai aqui

Você pode usar o exemplo a seguir para testar o código.
Vamos começar com um `Rectangle` quadrado de largura e altura `100`.

In [76]:
corner = Point(20, 20)
rectangle = Rectangle(100, 100, corner)

O código a seguir deve criar um `Circle` que caiba dentro do quadrado.

In [77]:
center = rectangle.midpoint()
radius = rectangle.height / 2

circle = Circle(center, radius)
print(circle)

Se tudo tiver funcionado corretamente, o código a seguir deve desenhar o círculo dentro do quadrado (tocando os quatro lados).

In [78]:
make_turtle(delay=0.01)

rectangle.draw()
circle.draw()

[Pense em Python: 3ª edição](https://allendowney.github.io/ThinkPython/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

Tradução educacional para o português brasileiro, sem fins lucrativos.

Licença do código: [Licença MIT](https://mit-license.org/)

Licença do texto: [Creative Commons Atribuição-NãoComercial-CompartilhaIgual 4.0 Internacional](https://creativecommons.org/licenses/by-nc-sa/4.0/)